In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 32
EPOCHS = 10

train_dir = 'data/train'
val_dir = 'data/val'

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)


In [ ]:
model = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D(2,2),
    
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # binary classification
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()


In [ ]:
history = model.fit(
    train_data,
    epochs=EPOCHS,
    validation_data=val_data
)


In [ ]:
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.title('Accuracy over Epochs')
plt.legend()
plt.show()


In [ ]:
val_data.reset()
preds = model.predict(val_data, verbose=1)
y_pred = (preds > 0.5).astype(int)
y_true = val_data.classes

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=['no_crack', 'crack']))


In [ ]:
model.save("crack_detector.h5")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('crack_detector.tflite', 'wb') as f:
    f.write(tflite_model)

print("✅ Model saved as crack_detector.tflite")
